In [1]:
# 스팸 메일 필터링 모델

import nltk
import numpy as np

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
docs = [
    'additional income',
    'best price',
    'big bucks',
    'cash bonus',
    'earn extra cash',
    'spring savings certificate',
    'valero gas marketing',
    'all domestic employees',
    'nominations for oct',
    'confirmation from spinner'
]

labels = np.array([1,1,1,1,1,0,0,0,0,0])

In [3]:
stop_words = set(stopwords.words('english'))

processed_docs = []

for doc in docs:

    tokens = word_tokenize(doc)

    # 구두점 제거
    words = [word.lower() for word in tokens if word.isalpha()]

    # 불용어 제거
    words = [word for word in words if word not in stop_words]

    processed_docs.append(" ".join(words))

print("전처리 결과")
print(processed_docs)

전처리 결과
['additional income', 'best price', 'big bucks', 'cash bonus', 'earn extra cash', 'spring savings certificate', 'valero gas marketing', 'domestic employees', 'nominations oct', 'confirmation spinner']


In [4]:
vocab_size = 50

encoded_docs = [
    one_hot(d, vocab_size)
    for d in processed_docs
]

print("\n인코딩 결과")
print(encoded_docs)


인코딩 결과
[[32, 5], [23, 24], [3, 28], [41, 13], [3, 25, 41], [26, 38, 36], [1, 4, 32], [11, 45], [2, 3], [38, 20]]


In [5]:
max_length = 4

padded_docs = pad_sequences(
    encoded_docs,
    maxlen=max_length,
    padding='post'
)

print("\n패딩 결과")
print(padded_docs)


패딩 결과
[[32  5  0  0]
 [23 24  0  0]
 [ 3 28  0  0]
 [41 13  0  0]
 [ 3 25 41  0]
 [26 38 36  0]
 [ 1  4 32  0]
 [11 45  0  0]
 [ 2  3  0  0]
 [38 20  0  0]]


In [6]:
model = Sequential()

model.add(
    Embedding(
        vocab_size,
        8,
        input_length=max_length
    )
)

model.add(Flatten())

model.add(
    Dense(
        1,
        activation='sigmoid'
    )
)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [7]:

model.fit(
    padded_docs,
    labels,
    epochs=50,
    verbose=1
)


Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 0.6961
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.5000 - loss: 0.6933
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.5000 - loss: 0.6905
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.5000 - loss: 0.6878
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5000 - loss: 0.6850
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.5000 - loss: 0.6823
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.6000 - loss: 0.6796
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.6000 - loss: 0.6768
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.7000 - loss: 0.6741
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.7000 - loss: 0.6714
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.7000 - loss: 0.6686
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.7000 - loss: 0.6659
Epo

In [8]:
loss, accuracy = model.evaluate(
    padded_docs,
    labels,
    verbose=0
)

print("\n정확도 =", accuracy)


정확도 = 1.0


In [9]:

test_doc = ['big income']

encoded_test = [
    one_hot(d, vocab_size)
    for d in test_doc
]

padded_test = pad_sequences(
    encoded_test,
    maxlen=max_length,
    padding='post'
)

prediction = model.predict(padded_test)

print("\n스팸 확률 =", prediction[0][0])

if prediction[0][0] > 0.5:
    print("스팸 메일입니다.")
else:
    print("정상 메일입니다.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step

스팸 확률 = 0.5911163
스팸 메일입니다.
